# 03 — Image Embeddings

Convert each of the 4,681 local product images into a numerical vector using a pretrained CLIP vision model.

CLIP (Contrastive Language-Image Pretraining) produces **image and text embeddings in the same shared semantic space**, which is exactly what we need for multimodal search later.

**Output files produced by this notebook:**
- `data/processed/image_embeddings.npy` — shape `(4681, 512)`, normalized float32 vectors
- `data/processed/image_embedding_index.csv` — maps each embedding row index → `pid`, `image_path`

## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

# Detect device — use GPU if available, otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## 2. Load ML-ready Dataset

We use `products_ml_ready.csv` which contains 4,681 products with valid local images.
We use `pid` as the product identifier — never the dataframe index.

In [2]:
df = pd.read_csv("../data/processed/products_ml_ready.csv")

print(f"Products loaded: {len(df)}")
assert len(df) == 4681, f"Expected 4681 products, got {len(df)}"

print(f"Columns: {df.columns.tolist()}")
df[["pid", "product_name", "main_category", "image_path"]].head(3)

Products loaded: 4681
Columns: ['pid', 'product_name', 'main_category', 'product_category_tree', 'brand', 'description', 'product_specifications', 'retail_price', 'discounted_price', 'image_path', 'combined_text', 'combined_text_clean']


,pid,product_name,main_category,image_path
0,LJGDYQ2HQQGVHHXG,Sukuma Women's Leggings,Clothing,../data/images/LJGDYQ2HQQGVHHXG.jpg
1,SHTE7ZPZZY64JF72,Kalrav Men's Solid Formal Shirt,Clothing,../data/images/SHTE7ZPZZY64JF72.jpg
2,TOPE94JH8ZFHZVTV,Noble Faith Casual Full Sleeve Solid Women's Top,Clothing,../data/images/TOPE94JH8ZFHZVTV.jpg


## 3. Verify Image Paths

Before embedding, confirm every image file exists and can be opened by Pillow.
We report valid vs invalid without deleting anything.

In [3]:
# image_path values are relative (../data/images/...) — resolve from notebook location
NOTEBOOK_DIR = Path(".").resolve()  # notebooks/

valid_rows = []
invalid_rows = []

for _, row in df.iterrows():
    abs_path = (NOTEBOOK_DIR / row["image_path"]).resolve()
    try:
        if not abs_path.exists():
            raise FileNotFoundError(f"File not found: {abs_path}")
        with Image.open(abs_path) as img:
            img.verify()  # checks file integrity without fully decoding
        valid_rows.append(row)
    except Exception as e:
        invalid_rows.append({"pid": row["pid"], "image_path": row["image_path"], "error": str(e)})

valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)

print(f"Total images:   {len(df)}")
print(f"Valid images:   {len(valid_df)}")
print(f"Invalid images: {len(invalid_rows)}")

if invalid_rows:
    print("\nInvalid files:")
    for r in invalid_rows[:10]:
        print(f"  pid={r['pid']} | {r['error']}")

Total images:   4681
Valid images:   4681
Invalid images: 0


## 4. Load CLIP Model

We use `openai/clip-vit-base-patch32` from Hugging Face.

- Image embedding dimension: **512**
- Text embedding dimension: **512** (same space — critical for multimodal search)
- Model size: ~340 MB download (cached after first run)

In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Loading CLIP model: {MODEL_NAME} ...")
clip_model = CLIPModel.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)

clip_model = clip_model.to(DEVICE)
clip_model.eval()  # inference mode — disables dropout etc.

print(f"Model loaded on: {DEVICE}")

Loading CLIP model: openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded on: cpu


## 5. Test Image Preprocessing and Embedding on a Small Sample

Run 5 images through the model before processing all 4,681.
Verify shape and that no errors occur.

In [5]:
sample_df = valid_df.head(5)

sample_images = []
for _, row in sample_df.iterrows():
    abs_path = (NOTEBOOK_DIR / row["image_path"]).resolve()
    img = Image.open(abs_path).convert("RGB")
    sample_images.append(img)

# Preprocess: CLIP expects specific pixel normalization
inputs = clip_processor(images=sample_images, return_tensors="pt", padding=True)
pixel_values = inputs["pixel_values"].to(DEVICE)

with torch.no_grad():
    # vision_model → pooler_output → visual_projection gives (N, 512) in the CLIP shared space
    vis_out = clip_model.vision_model(pixel_values=pixel_values)
    image_features = clip_model.visual_projection(vis_out.pooler_output)

print(f"Sample embeddings shape: {image_features.shape}")
print(f"Expected: ({len(sample_images)}, 512)")
assert image_features.shape[0] == len(sample_images), "Unexpected number of embeddings!"
assert image_features.shape[1] == 512, "Unexpected embedding dimension!"
print("Sample test passed.")

Sample embeddings shape: torch.Size([5, 512])
Expected: (5, 512)
Sample test passed.


## 6. Generate Embeddings for All 4,681 Images

We process images in batches to avoid loading everything into RAM at once.
- Batch size: 32 (safe for RTX 2050 4GB / CPU)
- Any image that fails to load is skipped and logged — it won't crash the whole run.
- We track `pid` and `image_path` for every successfully embedded product.

In [6]:
BATCH_SIZE = 32

all_embeddings = []
embedding_index = []  # list of {embedding_index, pid, image_path}
failures = []
current_idx = 0

rows = list(valid_df.iterrows())

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc="Embedding images"):
    batch_rows = rows[batch_start: batch_start + BATCH_SIZE]

    batch_images = []
    batch_meta = []

    for _, row in batch_rows:
        abs_path = (NOTEBOOK_DIR / row["image_path"]).resolve()
        try:
            img = Image.open(abs_path).convert("RGB")
            batch_images.append(img)
            batch_meta.append({"pid": row["pid"], "image_path": row["image_path"]})
        except Exception as e:
            failures.append({"pid": row["pid"], "image_path": row["image_path"], "error": str(e)})

    if not batch_images:
        continue

    try:
        inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True)
        pixel_values = inputs["pixel_values"].to(DEVICE)

        with torch.no_grad():
            vis_out = clip_model.vision_model(pixel_values=pixel_values)
            features = clip_model.visual_projection(vis_out.pooler_output)  # (batch, 512)

        features = features.cpu().float().numpy()
        all_embeddings.append(features)

        for meta in batch_meta:
            embedding_index.append({
                "embedding_index": current_idx,
                "pid": meta["pid"],
                "image_path": meta["image_path"]
            })
            current_idx += 1

    except Exception as e:
        for meta in batch_meta:
            failures.append({"pid": meta["pid"], "image_path": meta["image_path"], "error": str(e)})

# Stack all batches into one matrix
image_embeddings = np.vstack(all_embeddings)  # (N, 512)

print(f"\nEmbeddings generated")
print(f"  Successful: {len(embedding_index)}")
print(f"  Failed:     {len(failures)}")
print(f"  Shape:      {image_embeddings.shape}")

Embedding images:   0%|          | 0/147 [00:00<?, ?it/s]


Embeddings generated
  Successful: 4681
  Failed:     0
  Shape:      (4681, 512)


## 7. Normalize Embeddings

L2-normalize each embedding vector so that:
- Cosine similarity = dot product (faster retrieval)
- All vectors lie on the unit hypersphere

This is consistent with the text embeddings (also normalized).

In [7]:
# L2 normalize: divide each row by its norm
norms = np.linalg.norm(image_embeddings, axis=1, keepdims=True)
image_embeddings_normalized = image_embeddings / np.clip(norms, a_min=1e-10, a_max=None)

# Verify norms are ~1.0 after normalization
post_norms = np.linalg.norm(image_embeddings_normalized, axis=1)
print(f"Post-normalization norms — min: {post_norms.min():.6f}, max: {post_norms.max():.6f}")
print("All vectors normalized.")

Post-normalization norms — min: 1.000000, max: 1.000000
All vectors normalized.


## 8. Save Embeddings and Index

Two files:
1. `image_embeddings.npy` — the embedding matrix, row order matches `image_embedding_index.csv`
2. `image_embedding_index.csv` — maps each row index → `pid`, `image_path`

In [8]:
PROCESSED_DIR = Path("../data/processed")

# Save embedding matrix
emb_path = PROCESSED_DIR / "image_embeddings.npy"
np.save(emb_path, image_embeddings_normalized)

# Save index mapping
index_path = PROCESSED_DIR / "image_embedding_index.csv"
index_df = pd.DataFrame(embedding_index)
index_df.to_csv(index_path, index=False)

emb_size_mb = os.path.getsize(emb_path) / (1024 * 1024)
print(f"Saved image_embeddings.npy  → {emb_path}  ({emb_size_mb:.1f} MB)")
print(f"Saved image_embedding_index.csv → {index_path}")
print(f"Index rows: {len(index_df)}")
index_df.head(3)

Saved image_embeddings.npy  → ..\data\processed\image_embeddings.npy  (9.1 MB)
Saved image_embedding_index.csv → ..\data\processed\image_embedding_index.csv
Index rows: 4681


,embedding_index,pid,image_path
0,0,LJGDYQ2HQQGVHHXG,../data/images/LJGDYQ2HQQGVHHXG.jpg
1,1,SHTE7ZPZZY64JF72,../data/images/SHTE7ZPZZY64JF72.jpg
2,2,TOPE94JH8ZFHZVTV,../data/images/TOPE94JH8ZFHZVTV.jpg


## 9. Validate Saved Embeddings

Reload from disk and confirm everything looks correct.

In [9]:
loaded_embeddings = np.load(emb_path)
loaded_index = pd.read_csv(index_path)

print("=== Validation ===")
print(f"Shape:             {loaded_embeddings.shape}")
print(f"Dtype:             {loaded_embeddings.dtype}")
print(f"Embedding rows:    {loaded_embeddings.shape[0]}")
print(f"Index rows:        {len(loaded_index)}")
print(f"NaN values:        {np.isnan(loaded_embeddings).sum()}")
print(f"Inf values:        {np.isinf(loaded_embeddings).sum()}")
print(f"Rows match index:  {loaded_embeddings.shape[0] == len(loaded_index)}")

assert loaded_embeddings.shape[0] == len(loaded_index), "Row count mismatch!"
assert np.isnan(loaded_embeddings).sum() == 0, "NaN values found!"
assert np.isinf(loaded_embeddings).sum() == 0, "Inf values found!"
print("\nAll validation checks passed.")

=== Validation ===
Shape:             (4681, 512)
Dtype:             float32
Embedding rows:    4681
Index rows:        4681
NaN values:        0
Inf values:        0
Rows match index:  True

All validation checks passed.


## 10. Image Similarity Sanity Check

Pick the first product as a query. Compare its embedding against all others using dot product (equivalent to cosine similarity since embeddings are normalized).

The query product itself should appear at rank 1 with similarity ≈ 1.0.

In [10]:
# Use product at index 0 as the query
query_idx = 0
query_embedding = loaded_embeddings[query_idx]  # shape (512,)

# Dot product with all embeddings (cosine similarity since normalized)
similarity_scores = loaded_embeddings @ query_embedding  # shape (N,)

# Get top 10 (includes the query itself at rank 1)
top_indices = np.argsort(similarity_scores)[::-1][:10]

# Build results table
results = []
for rank, idx in enumerate(top_indices, 1):
    row = loaded_index.iloc[idx]
    product_info = df[df["pid"] == row["pid"]].iloc[0]
    results.append({
        "rank": rank,
        "similarity": round(float(similarity_scores[idx]), 4),
        "pid": row["pid"],
        "product_name": product_info["product_name"][:50],
        "main_category": product_info["main_category"],
        "image_path": row["image_path"]
    })

results_df = pd.DataFrame(results)

query_product = df[df["pid"] == loaded_index.iloc[query_idx]["pid"]].iloc[0]
print(f"Query product: {query_product['product_name']} ({query_product['main_category']})")
print(f"Query image:   {loaded_index.iloc[query_idx]['image_path']}\n")
print("Top 10 visually similar products:")
results_df

Query product: Sukuma Women's Leggings (Clothing)
Query image:   ../data/images/LJGDYQ2HQQGVHHXG.jpg

Top 10 visually similar products:


,rank,similarity,pid,product_name,main_category,image_path
0,1,1.0000,LJGDYQ2HQQGVHHXG,Sukuma Women's Leggings,Clothing,../data/images/LJGDYQ2HQQGVHHXG.jpg
1,2,0.9184,LJGE4YPSH5NK3GMY,Kjaggs Women's Leggings,Clothing,../data/images/LJGE4YPSH5NK3GMY.jpg
2,3,0.8940,LJGEJQ4H82QG64PN,medha Women's Multicolor Leggings,Clothing,../data/images/LJGEJQ4H82QG64PN.jpg
3,4,0.8938,LJGEDPJAHHZHR7UB,Kjaggs Women's Leggings,Clothing,../data/images/LJGEDPJAHHZHR7UB.jpg
4,5,0.8929,LJGEG52SRXN5JXFE,Famaya Girl's Leggings,Clothing,../data/images/LJGEG52SRXN5JXFE.jpg
5,6,0.8782,LJGE8GG2XTA4YXH4,NE Women's Leggings,Clothing,../data/images/LJGE8GG2XTA4YXH4.jpg
6,7,0.8721,LJGE8MZQEVAFUQ7G,Kimmy Women's Leggings,Clothing,../data/images/LJGE8MZQEVAFUQ7G.jpg
7,8,0.8688,ACBEHFPVV7PYCVHR,FIFO Bottom Women's Combo,Clothing,../data/images/ACBEHFPVV7PYCVHR.jpg
8,9,0.8683,LJGE8NS7A3RNHBNF,NE Women's Leggings,Clothing,../data/images/LJGE8NS7A3RNHBNF.jpg
9,10,0.8642,LJGE9YJHTJZCQ5VP,Escocer Women's Leggings,Clothing,../data/images/LJGE9YJHTJZCQ5VP.jpg


## Summary

In [11]:
print("=" * 55)
print("IMAGE EMBEDDING PIPELINE — FINAL REPORT")
print("=" * 55)
print(f"1. Products processed:          {len(valid_df)}")
print(f"2. Successful image embeddings: {loaded_embeddings.shape[0]}")
print(f"3. Failures:                    {len(failures)}")
print(f"4. Embedding dimension:         {loaded_embeddings.shape[1]}")
print(f"5. Embedding dtype:             {loaded_embeddings.dtype}")
print(f"6. Embeddings normalized:       Yes (L2)")
print(f"7. Saved embeddings:            {emb_path}")
print(f"8. Saved index mapping:         {index_path}")
print(f"9. Sanity check rank-1 score:   {results_df.iloc[0]['similarity']} (should be ~1.0)")
print("=" * 55)

IMAGE EMBEDDING PIPELINE — FINAL REPORT
1. Products processed:          4681
2. Successful image embeddings: 4681
3. Failures:                    0
4. Embedding dimension:         512
5. Embedding dtype:             float32
6. Embeddings normalized:       Yes (L2)
7. Saved embeddings:            ..\data\processed\image_embeddings.npy
8. Saved index mapping:         ..\data\processed\image_embedding_index.csv
9. Sanity check rank-1 score:   1.0 (should be ~1.0)
